# KoHRM SFT/LoRA Data Runbook

이 노트북은 Colab 또는 로컬에서 KoHRM SFT/LoRA prepared dataset repo를 확인하고, 어떤 subset으로 LoRA를 돌릴지 결정하기 위한 점검용입니다. T4에서 1.4B LoRA 학습을 실제로 돌리는 노트북이 아닙니다. H200 서버에서는 아래 명령을 repo 루트에서 실행합니다.

## 1. Install dependencies

In [ ]:
!pip -q install -U huggingface_hub hf_transfer numpy "tokenizers>=0.22.0,<0.23.1"

## 2. Dataset repo settings

In [ ]:
import os
import json
from pathlib import Path

from huggingface_hub import HfApi, snapshot_download

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

DATASET_REPO_ID = "LLM-OS-Models/KoHRM-Text-1.4B-sft-lora-data"
REVISION = "main"

api = HfApi()
info = api.dataset_info(DATASET_REPO_ID, revision=REVISION)
print("dataset sha:", info.sha)
print("num siblings:", len(info.siblings))
for item in info.siblings[:30]:
    print(item.rfilename)

## 3. Recommended subsets

In [ ]:
SUBSETS = {
    "behavior-mini": {
        "folder": "kohrm_sft_behavior_mini_v1",
        "tokens": 60000387,
        "samples": 61810,
        "purpose": "quick behavior smoke: Korean answer style, JSON/tool-call form, terminal command behavior",
    },
    "korean-domain": {
        "folder": "kohrm_sft_korean_domain_core_v1",
        "tokens": 100000654,
        "samples": 219072,
        "purpose": "Korean legal/admin-rule extraction and Korean finance QA style",
    },
    "terminal-tool": {
        "folder": "kohrm_sft_terminal_tool_core_v1",
        "tokens": 165007375,
        "samples": 55934,
        "purpose": "terminal trajectories, tool-call JSON, SWE/code workflow",
    },
    "behavior-core": {
        "folder": "kohrm_sft_behavior_core_v1",
        "tokens": 285008218,
        "samples": 291382,
        "purpose": "broad behavior alignment mix",
    },
}

print(f"{'name':<16} {'folder':<40} {'tokens':>12} {'samples':>10}")
for name, spec in SUBSETS.items():
    print(f"{name:<16} {spec['folder']:<40} {spec['tokens']:>12,} {spec['samples']:>10,}")

## 4. Download metadata only

전체 `tokens.npy`까지 받으면 수 GB가 필요합니다. Colab에서는 먼저 metadata만 받는 것이 안전합니다.

In [ ]:
metadata_dir = Path(snapshot_download(
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    revision=REVISION,
    allow_patterns=[
        "README.md",
        "*/metadata.json",
        "*/tokenizer_info.json",
        "*/sample_stats.json",
        "*/merge_stats.json",
    ],
    max_workers=8,
))

print("metadata cache:", metadata_dir)
for name, spec in SUBSETS.items():
    folder = metadata_dir / spec["folder"]
    print("\n==", name, "==")
    for filename in ["metadata.json", "sample_stats.json", "merge_stats.json"]:
        path = folder / filename
        if path.exists():
            print(filename, json.loads(path.read_text()))

## 5. Optional: download one full subset

실제 학습 서버에서는 이미 `/home/work/.data/hrm_text_prepared`에 있어야 합니다. 다른 PC에서 이어서 학습하려면 필요한 subset만 내려받습니다.

In [ ]:
DOWNLOAD_FULL_SUBSET = False
SELECTED = "kohrm_sft_behavior_mini_v1"

if DOWNLOAD_FULL_SUBSET:
    subset_dir = Path(snapshot_download(
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        revision=REVISION,
        allow_patterns=[f"{SELECTED}/**"],
        max_workers=8,
    )) / SELECTED
    print("downloaded subset:", subset_dir)
    for path in sorted(subset_dir.iterdir()):
        if path.is_file():
            print(path.name, round(path.stat().st_size / 2**20, 2), "MiB")
else:
    print("Set DOWNLOAD_FULL_SUBSET=True to download one complete prepared subset.")

## 6. H200 LoRA commands

아래 명령은 Colab이 아니라 H200 학습 서버에서 실행합니다. 현재 pretraining을 방해하지 않도록 별도 시점에 실행합니다.

In [ ]:
BASE = "/home/work/.data/hrm_text_checkpoints/KoHRM-Text-1.4B-stage4b-korean-tool-finance-repeat-gbs180"
commands = {
    "phase1": f"export RESUME_FROM={BASE} && bash scripts/run_kohrm_lora_experiments.sh phase1",
    "behavior-mini": f"export RESUME_FROM={BASE} && bash scripts/run_kohrm_lora_experiments.sh behavior-mini",
    "korean-domain": f"export RESUME_FROM={BASE} && bash scripts/run_kohrm_lora_experiments.sh korean-domain",
    "terminal-tool": f"export RESUME_FROM={BASE} && bash scripts/run_kohrm_lora_experiments.sh terminal-tool",
    "behavior-core": f"export RESUME_FROM={BASE} && bash scripts/run_kohrm_lora_experiments.sh behavior-core",
}
for name, command in commands.items():
    print("\n#", name)
    print(command)

## 7. Decision rule

현재 공개 checkpoint에서 반복, 영어 agent reasoning, JSON fidelity 문제가 보이면 `behavior-mini`를 먼저 돌립니다. 한국어 도메인 응답이 계속 약하면 `korean-domain`, 명령만 답하기/tool-call JSON이 약하면 `terminal-tool`을 이어서 돌립니다. 이 세 후보가 좋아지면 RL 또는 preference optimization으로 넘어가고, 여전히 전반 행동이 흔들리면 `behavior-core` short SFT/LoRA를 사용합니다.